# Abigale notebook

In [ ]:
import ocean_skill as osk

## Set up model catalog

This saves the kerchunk representation of the model output at model_base as a parquet file

In [ ]:
# model output is in
model_base = "/anvil/scratch/x-smaticka/sims_runtime/cstar/pac_dt_ramp/pac_dt_ramp/tasks/third_1wks/joined_output/"
# grid is at 
grid_loc = "/anvil/scratch/x-smaticka/sims_runtime/cstar/pac_dt_ramp/pac_dt_ramp/tasks/third_1wks/input/input_datasets/PACMED12km_spec_config_WithRivers_grid.nc"
# name of catalog - also using for the dataset name since there is only one in here
cat_name = "pac_dt_ramp"

refs = build_kerchunk({cat_name: "output_rst.*.nc"},
                      root=model_base, 
                      grid=grid_loc, 
                      keep="latest-per-file",  # use this for restart files to keep the main time step
                      out_dir=model_base)
build_catalog(refs, f"catalogs/{cat_name}.yaml", title=cat_name)

## WOA Nutrients - Monthly Means

Monthly being compared to daily at surface

### Surface

In [ ]:
# these are being referred to by nicknames
vars_to_plot = ["nitrate", "phosphate", "silicate", "oxygen"]

physics = osk.compare(
    aggregate={"time": {"resample": "1MS", "reduce": "mean"}},
    reference=["woa23_nitrate_month01", "woa23_phosphate_month01", "woa23_silicate_month01", "woa23_oxygen_month01"],
    test=cat_name,  # using cat_name here because it is used for the model/dataset name too
    variables=vars_to_plot,
    depths=["surface"],
)
fig = physics.plot()
# fig = physics.plot(title="ROMS GOM vs. WOA (Surface, Jan 2001)")

### 100m Depth

In [ ]:
physics = osk.compare(
    aggregate={"time": {"resample": "1MS", "reduce": "mean"}},
    reference=["woa23_nitrate_month01", "woa23_phosphate_month01", "woa23_silicate_month01", "woa23_oxygen_month01"],
    test=cat_name,  # using cat_name here because it is used for the model/dataset name too
    variables=vars_to_plot,
    depths=[100],
)
fig = physics.plot()
# fig = physics.plot(title="ROMS GOM vs. WOA (Surface, Jan 2001)")

## GLODAP Alk and DIC - Climatology

Average our model output to do this comparison - can't compute a real climatology for the model output currently.

### Surface

In [ ]:
vars_to_plot = ["alkalinity", "DIC"]

physics = osk.compare(
    aggregate={"time": "mean"},
    reference=["glodap"],
    test=cat_name,
    variables=vars_to_plot,
    depths=["surface"],
)
fig = physics.plot()
# fig = physics.plot(title="ROMS GOM vs. GLODAP (Surface, Climatology)", **plot_kwargs)

### 100m Depth

In [ ]:
physics = osk.compare(
    aggregate={"time": "mean"},
    reference=["glodap"],
    test=cat_name,
    variables=vars_to_plot,
    depths=[100],
)
fig = physics.plot()
# fig = physics.plot(title="ROMS GOM vs. GLODAP (Surface, Climatology)", **plot_kwargs)

### Could also compare by depth

In [ ]:
physics = osk.compare(
    aggregate={"time": "mean"},
    reference=["glodap"],
    test=cat_name,
    variables="alkalinity",
    depths=["surface", "100"],
)
fig = physics.plot(shared_limits=True,)

## Satellite Chlorophyll

In [ ]:
osk.find(variable="chlorophyll")

In [ ]:
osk.describe('erdMWchla1day')

In [ ]:
osk.describe('erdMH1chla1day_R2022SQ')

In [ ]:
osk.find(name="modis", variable="chlorophyll", text="jan")

In [ ]:
CHL = "mass_concentration_of_chlorophyll_a_in_sea_water"
plot_kwargs = {"colorbar_kwargs": {"pad": 0.05, "shrink": 0.65}, "metrics_kwargs": {"fontsize": 10}}


plot_kwargs = {"colorbar_kwargs": {"pad": 0.03, "shrink": 0.8,  "label_size": 9}, "figsize": (12, 3),
               "metrics_kwargs": {"fontsize": 10},
               "suptitle_kwargs": {"fontsize": 13},
               "title_kwargs": {"fontsize": 11},
               "row_label_kwargs": {"fontsize": 11},
               "tick_label_kwargs": {"size": 9},
               }


chl = osk.compare(
    reference=['erdMWchla1day'],
    test=cat_name,
    variables=[{"sum": ["spChl", "diatChl"], "standard_name": CHL}],
    select={"T": "2012-01", "Z": {"min": 0, "max": 10}},
    aggregate={"T": "mean", "Z": "mean"},
)

fig = chl.plot(title="ROMS GOM (integrated over top 10 m) vs. MODIS Aqua, January 2003 mean", **plot_kwargs)

## Summary Diagrams

Show all comparisons on one Taylor and one Target diagram.

In [ ]:
grouped = osk.compare(
    aggregate={"time": "mean"},
    reference=["woa23_nitrate_month01", "woa23_phosphate_month01", "woa23_silicate_month01", "woa23_oxygen_month01", "glodap"],
    test="GOM offline run:GOM_bgc",
    variables=[NITRATE, PHOSPHATE, SILICATE, OXYGEN, ALKALINITY, DIC],
    depths=("surface", 100),
)
grouped.summary(
    title="ROMS vs Multiple Sources — colour by variable, marker by depth",
    color_by="variable", marker_by="depth", labels="legend", figsize=(12, 6)
);

## Others to Demo

### OceanSODA

In [ ]:
osk.find(text="soda")

In [ ]:
print(osk.describe("OceanSODA2"))

In [ ]:
osk.read("OceanSODA2")

In [ ]:
ALKALINITY = "alkalinity"

plot_kwargs = {"colorbar_kwargs": {"pad": 0.03, "shrink": 0.8,  "label_size": 9}, "figsize": (12, 6),
               "metrics_kwargs": {"fontsize": 10},
               "suptitle_kwargs": {"fontsize": 13},
               "title_kwargs": {"fontsize": 11},
               "row_label_kwargs": {"fontsize": 11},
               "tick_label_kwargs": {"size": 9},
               }


physics = osk.compare(
    aggregate={"time": "mean"},
    reference=["glodap", "OceanSODA2"],
    test="GOM offline run:GOM_bgc",
    variables=[ALKALINITY],
    depths=["surface"],
)
fig = physics.plot(title="ROMS GOM vs. GLODAP (Surface, Climatology)", **plot_kwargs, shared_limits=True,)